# 日期时间时区与时间差

学习目标：解析日期时间，按时间筛选记录，统一时区并计算间隔，识别缺失、夏令时和时间精度边界。

前置知识：日期、时间、时区的基本含义，索引与类型转换。

运行环境：Python 3.12、pandas 3；本章按 pandas 3 的时间单位推断规则编写。

环境准备：[环境配置与运行](README.md)

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

## 1 计算处理时长

下面是两条自制任务记录。输入采用同一格式，均表示同一地点的本地时间，且不跨夏令时切换。

to_datetime 将文本转为日期时间。format 中的 %Y、%m、%d 分别表示年、月、日，%H、%M 表示 24 小时制的小时、分钟。两列相减得到时间差 Timedelta；通过 dt.total_seconds() 换算总秒数，再除以 60 得到分钟数。

In [1]:
import pandas as pd

tasks = pd.DataFrame({
    "task": ["A", "B"],
    "start_text": ["2026-09-20 09:00", "2026-09-20 09:40"],
    "end_text": ["2026-09-20 09:25", "2026-09-20 10:20"],
})
tasks["start"] = pd.to_datetime(tasks["start_text"], format="%Y-%m-%d %H:%M")
tasks["end"] = pd.to_datetime(tasks["end_text"], format="%Y-%m-%d %H:%M")
tasks["elapsed"] = tasks["end"] - tasks["start"]
tasks["minutes"] = tasks["elapsed"].dt.total_seconds() / 60
print(tasks[["task", "elapsed", "minutes"]])
print(tasks[["start", "elapsed"]].dtypes)
# A、B 分别用时 25、40 分钟；日期与时间差列都有自己的时间单位。

  task         elapsed  minutes
0    A 0 days 00:25:00     25.0
1    B 0 days 00:40:00     40.0
start       datetime64[us]
elapsed    timedelta64[us]
dtype: object


## 2 单个时间与时间索引

Timestamp 表示单个日期时间；DatetimeIndex 保存一组日期时间，可作为行标签。to_datetime 接收单个字符串时返回 Timestamp，接收列表时返回 DatetimeIndex，接收 Series 时返回保留其索引的 Series。

没有时区信息的时间称为无时区时间（tz-naive）；仅凭这种值不能确定全球统一的时刻。

In [2]:
instant = pd.Timestamp("2026-09-20 09:00")
times = pd.to_datetime(["2026-09-20 09:00", "2026-09-20 10:00"])
counts = pd.Series([3, 5], index=times, name="count")
print(type(instant).__name__, type(times).__name__)
print(counts.loc[instant])
print(times.dtype)
# 标签 09:00 对应 3；索引 dtype 中未包含时区。

Timestamp DatetimeIndex
3
datetime64[us]


## 3 解析失败与 NaT

errors="raise" 是默认规则，无法解析时抛出异常。errors="coerce" 将失败项转为 NaT，NaT 是日期时间和时间差的缺失标记。使用 isna() 找出缺失，并保留原文本，才能区分原本缺失与转换失败。

固定格式的输入应明确给出 format。日期顺序不清楚时，dayfirst 只是解析偏好，不能代替严格的输入格式约定。

In [3]:
raw = pd.Series(["2026/09/20", "2026/02/30", None], index=["A", "B", "C"])
parsed = pd.to_datetime(raw, format="%Y/%m/%d", errors="coerce")
failed = raw.notna() & parsed.isna()
print(pd.DataFrame({"raw": raw, "parsed": parsed, "failed": failed}))
# B 是日期非法导致的失败；C 原本缺失，两项转换后均为 NaT。
try:
    pd.to_datetime("2026/02/30", format="%Y/%m/%d", errors="raise")
except ValueError:
    print("ValueError：输入不是有效日期")
else:
    raise AssertionError("应拒绝不存在的日期")

          raw     parsed  failed
A  2026/09/20 2026-09-20   False
B  2026/02/30        NaT    True
C         NaN        NaT   False
ValueError：输入不是有效日期


## 4 数字时间的单位与起点

数字没有自带日期含义。unit 指定数字的单位，origin 指定起点；origin="unix" 表示 1970-01-01。下面的 60 按秒解释是 1 分钟，按毫秒解释只有 0.06 秒。

也可以约定自定义起点，例如用整数天数保存相对项目启动日的日期。数字输入应明确单位，不依赖默认解释。

In [4]:
print(pd.to_datetime([0, 60], unit="s", origin="unix"))
print(pd.to_datetime([0, 60], unit="ms", origin="unix"))
project_days = pd.to_datetime([0, 2], unit="D", origin="2026-09-20")
print(project_days)
# 项目相对日期是 09-20、09-22；D 是输入单位，结果 dtype 不一定以 D 存储。

DatetimeIndex(['1970-01-01 00:00:00', '1970-01-01 00:01:00'], dtype='datetime64[s]', freq=None)
DatetimeIndex(['1970-01-01 00:00:00', '1970-01-01 00:00:00.060000'], dtype='datetime64[ms]', freq=None)
DatetimeIndex(['2026-09-20', '2026-09-22'], dtype='datetime64[s]', freq=None)


## 5 生成规则时间点

date_range 返回 DatetimeIndex。常用 start、end、freq 指定起止与频率，或用 start、periods、freq 指定起点、数量与频率。30min 表示每 30 分钟一个时间点。

inclusive 控制端点：both 包含两端，left 只包含左端，right 只包含右端，neither 排除两端。端点仍须符合所设频率；inclusive 不会额外插入偏离频率的时间点。unit 可显式指定结果的存储单位。

In [5]:
for boundary in ["both", "left", "right", "neither"]:
    slots = pd.date_range(
        "2026-09-20 09:00", "2026-09-20 10:00",
        freq="30min", inclusive=boundary, unit="s",
    )
    print(boundary, slots.strftime("%H:%M").tolist())
print(slots.dtype)
# 四种边界分别产生 3、2、2、1 个时间点；这里显式采用秒单位。

both ['09:00', '09:30', '10:00']
left ['09:00', '09:30']
right ['09:30', '10:00']
neither ['09:30']
datetime64[s]


## 6 提取字段与时间切片

Series 的 dt 访问器对整列提取日期字段。下面继续使用 tasks，dt.hour 得到开始时间的小时，dt.strftime 按指定格式生成展示文本；展示文本不能替代日期类型进行时间运算。

In [6]:
print(tasks["start"].dt.hour)
print(tasks["start"].dt.strftime("%Y-%m-%d"))
# 两条任务的小时均为 9；strftime 返回字符串 Series，原 start 列仍为日期类型。

0    9
1    9
Name: start, dtype: int32
0    2026-09-20
1    2026-09-20
Name: start, dtype: str


将时间列设为索引并排序后，可以用 loc 按时间筛选。日期字符串可以表示整天，起止标签切片包含两端。精确的 Timestamp 表示单个时间标签；日期字符串的范围选择与它不同。

In [7]:
readings = pd.Series(
    [12, 10, 11],
    index=pd.to_datetime([
        "2026-09-21 00:00", "2026-09-20 09:00", "2026-09-20 17:00",
    ]),
    name="temperature",
).sort_index()
print(readings.loc["2026-09-20"])
print(readings.loc["2026-09-20 17:00":"2026-09-21 00:00"])
# 整天筛选包含 09:00 和 17:00；第二个切片包含值 11、12，包括午夜端点。

2026-09-20 09:00:00    10
2026-09-20 17:00:00    11
Name: temperature, dtype: int64
2026-09-20 17:00:00    11
2026-09-21 00:00:00    12
Name: temperature, dtype: int64


## 7 时间单位、精度与范围

pandas 3 会根据输入推断日期时间和时间差的存储单位。字符串日期通常使用微秒，包含更细的小数秒时可以使用纳秒；数字输入也受指定 unit 影响。应查看实际 dtype，而不是把所有日期都当作 datetime64[ns]。

支持的存储单位包括 s（秒）、ms（毫秒）、us（微秒）、ns（纳秒）。时间单位越细，同样的 64 位整数能覆盖的范围越小；纳秒范围约为 1677 年至 2262 年，这不是其他单位的统一边界。

In [8]:
ordinary = pd.to_datetime(["2026-09-20"])
precise = pd.to_datetime(["2026-09-20 00:00:00.123456789"])
seconds = pd.to_datetime([0, 1], unit="s")
print(ordinary.dtype, precise.dtype, seconds.dtype)
far = pd.Timestamp("2500-01-01")
print(far, far.unit)
# 三组索引依次为 us、ns、s；2500 年可以用此处推断的 us 表示。
try:
    far.as_unit("ns")
except pd.errors.OutOfBoundsDatetime:
    print("OutOfBoundsDatetime：转成纳秒后超出范围")
else:
    raise AssertionError("2500 年不能用纳秒时间戳表示")

datetime64[us] datetime64[ns] datetime64[s]
2500-01-01 00:00:00 us
OutOfBoundsDatetime：转成纳秒后超出范围


as_unit 可以转换存储单位，但更粗的单位可能丢失小数部分。round_ok=False 要求转换不能损失精度，否则抛出异常；它不会自动寻找另一种合适的单位。

In [9]:
fine = pd.Timestamp("2026-09-20 00:00:00.123456789")
try:
    fine.as_unit("us", round_ok=False)
except ValueError:
    print("ValueError：微秒不能完整保留这 9 位小数秒")
else:
    raise AssertionError("应发现精度损失")
print(fine.as_unit("us"))
# 默认允许转换，此例末尾的 789 纳秒被丢弃，只保留 .123456。

ValueError：微秒不能完整保留这 9 位小数秒
2026-09-20 00:00:00.123456


## 8 赋予时区与转换时区

tz_localize 为无时区时间指定它原本所属的时区，保留钟面上的小时、分钟；tz_convert 把已有时区的同一时刻改用另一个时区表示。对 Series 中的时间值使用 dt.tz_localize 和 dt.tz_convert，对 DatetimeIndex 直接调用。

下面明确约定输入是上海本地时间。必须先附上 Asia/Shanghai，再转为 UTC；时区转换前后，实际时刻不变。

In [10]:
local_clock = pd.to_datetime(["2026-09-20 09:00", "2026-09-20 10:00"])
shanghai = local_clock.tz_localize("Asia/Shanghai")
utc_times = shanghai.tz_convert("UTC")
print(shanghai)
print(utc_times)
print(shanghai[0] == utc_times[0])
# 09:00 +08:00 对应 01:00 +00:00，比较结果为 True。
try:
    local_clock.tz_convert("UTC")
except TypeError:
    print("TypeError：无时区时间需要先确定所属时区")
else:
    raise AssertionError("无时区输入不应直接转换时区")

DatetimeIndex(['2026-09-20 09:00:00+08:00', '2026-09-20 10:00:00+08:00'], dtype='datetime64[us, Asia/Shanghai]', freq=None)
DatetimeIndex(['2026-09-20 01:00:00+00:00', '2026-09-20 02:00:00+00:00'], dtype='datetime64[us, UTC]', freq=None)
True
TypeError：无时区时间需要先确定所属时区


## 9 将不同偏移统一为 UTC

to_datetime 的 utc=True 将带时区或 UTC 偏移的输入转换为 UTC；对无时区输入则直接解释为 UTC。因此，上海本地钟面时间不能直接用 utc=True 代替时区本地化。

pandas 3 对混合 UTC 偏移且未指定 utc=True 的输入会报错。format 中的 %z 用来解析 UTC 偏移。下面的 +0200、+0100 分别表示比 UTC 快 2 小时、1 小时；它们不是两个时区的完整历史规则。

In [11]:
offset_text = ["2020-10-25 02:00 +0200", "2020-10-25 04:00 +0100"]
try:
    pd.to_datetime(offset_text, format="%Y-%m-%d %H:%M %z")
except ValueError:
    print("ValueError：混合偏移需要明确统一为 UTC")
else:
    raise AssertionError("pandas 3 应拒绝未统一的混合偏移")
unified = pd.to_datetime(offset_text, format="%Y-%m-%d %H:%M %z", utc=True)
print(unified)
print(unified[1] - unified[0])
# UTC 时间为 00:00、03:00，真实间隔为 3 小时。

ValueError：混合偏移需要明确统一为 UTC
DatetimeIndex(['2020-10-25 00:00:00+00:00', '2020-10-25 03:00:00+00:00'], dtype='datetime64[us, UTC]', freq=None)
0 days 03:00:00


## 10 夏令时的重复时刻

夏令时（DST）结束时，时钟回拨，某些本地时间会出现两次。例如 CET 在 2018-10-28 的 02:30 对应两个不同的 UTC 时刻。默认 ambiguous="raise" 拒绝这种不确定输入。

若原记录明确区分了两次出现，可以给 ambiguous 布尔列表：True 表示夏令时，False 表示标准时。也可用 "NaT" 标记无法判断的项；"infer" 根据顺序尝试推断，不能保证对任意输入都成功。

In [12]:
repeated = pd.to_datetime(["2018-10-28 02:30", "2018-10-28 02:30"])
try:
    repeated.tz_localize("CET")
except ValueError:
    print("ValueError：02:30 在回拨时出现两次")
else:
    raise AssertionError("应要求处理歧义时间")
resolved = repeated.tz_localize("CET", ambiguous=[True, False])
print(resolved)
print(resolved.tz_convert("UTC"))
print(resolved[1] - resolved[0])
# 相同的 02:30 分别为 +02:00 和 +01:00，两次事件实际相隔 1 小时。

ValueError：02:30 在回拨时出现两次
DatetimeIndex(['2018-10-28 02:30:00+02:00', '2018-10-28 02:30:00+01:00'], dtype='datetime64[us, CET]', freq=None)
DatetimeIndex(['2018-10-28 00:30:00+00:00', '2018-10-28 01:30:00+00:00'], dtype='datetime64[us, UTC]', freq=None)
0 days 01:00:00


## 11 夏令时中不存在的时刻

夏令时开始时，时钟向前跳过一段时间。Europe/Warsaw 的 2015-03-29 02:30 就不存在，默认 nonexistent="raise" 抛出异常。

只有业务允许修改记录时，才用 nonexistent="shift_forward" 推到最近的有效时刻；若无法推定原意，可用 "NaT" 保留为缺失。前者改变了输入所表达的钟面时间，不能当作原始事实。

In [13]:
missing_clock = pd.to_datetime(["2015-03-29 02:30", "2015-03-29 03:30"])
try:
    missing_clock.tz_localize("Europe/Warsaw")
except ValueError:
    print("ValueError：跳过的 02:30 不存在")
else:
    raise AssertionError("应要求处理不存在的时间")
print(missing_clock.tz_localize("Europe/Warsaw", nonexistent="shift_forward"))
print(missing_clock.tz_localize("Europe/Warsaw", nonexistent="NaT"))
# 第一种处理把 02:30 改到 03:00；第二种将它变成 NaT；原来的 03:30 保留。

ValueError：跳过的 02:30 不存在
DatetimeIndex(['2015-03-29 03:00:00+02:00', '2015-03-29 03:30:00+02:00'], dtype='datetime64[us, Europe/Warsaw]', freq=None)
DatetimeIndex(['NaT', '2015-03-29 03:30:00+02:00'], dtype='datetime64[us, Europe/Warsaw]', freq=None)


## 12 构造时间差

Timedelta 表示固定时长，to_timedelta 用于批量转换。数字输入应给出 unit，字符串则在内容中注明单位；errors="coerce" 将非法项变为 NaT。

这里 1 天表示 24 小时。月份、年份的长度不固定，不能把 "1M" 当作固定时长传入；按日历移动日期需要时间偏移。

In [14]:
waits = pd.to_timedelta(pd.Series([30, 90]), unit="s")
print(waits)
print(pd.to_timedelta(["1h 30min", "bad"], errors="coerce"))
print(pd.Timestamp("2026-09-20 09:00") + pd.Timedelta(minutes=30))
# 30、90 秒保持秒单位；文本的第二项变成 NaT；09:00 加 30 分钟得到 09:30。
try:
    pd.to_timedelta("1M")
except ValueError:
    print("ValueError：月份不是固定时长")
else:
    raise AssertionError("应拒绝含糊的月份时长")

0   0 days 00:00:30
1   0 days 00:01:30
dtype: timedelta64[s]
TimedeltaIndex(['0 days 01:30:00', NaT], dtype='timedelta64[us]', freq=None)
2026-09-20 09:30:00
ValueError：月份不是固定时长


## 13 总秒数与分量

total_seconds() 用于把完整时长换算成秒；seconds 属性只是减去整天后的秒数，不是总秒数。components 把显示出的时长拆成天、小时、分钟等字段。

负时长也会规范化为天数和非负的日内分量，例如负 1 小时显示为负 1 天加 23 小时。因此计算总耗时不要直接取 seconds 或 components 中的一列。

In [15]:
duration = pd.Timedelta(days=1, hours=2, minutes=3)
print(duration.total_seconds(), duration.seconds)
print(duration.components)
negative = pd.Timedelta(hours=-1)
print(negative)
print(negative.total_seconds(), negative.seconds)
# 1 天 2 小时 3 分钟共 93780 秒，seconds 只为 7380。
# 负 1 小时的总秒数为 -3600，日内 seconds 却是 82800。

93780.0 7380
Components(days=1, hours=2, minutes=3, seconds=0, milliseconds=0, microseconds=0, nanoseconds=0)
-1 days +23:00:00
-3600.0 82800


## 14 选学：月份区间
Period 表示带频率的一个时间区间，例如整个月，而非某个单独时刻。freq="M" 表示月度区间，加 1 前进一个月；start_time 可取得该区间的起点 Timestamp。

这里的 M 是 Period 的月频率；date_range 若要产生月末时间点，应使用 ME。频率名称须结合 API 理解。

In [16]:
month = pd.Period("2026-02", freq="M")
print(month, month + 1)
print(month.start_time)
print(pd.date_range("2026-02-01", periods=2, freq="ME"))
# Period 分别表示 2 月、3 月整段；月末时间点分别是 02-28、03-31。

2026-02 2026-03
2026-02-01 00:00:00
DatetimeIndex(['2026-02-28', '2026-03-31'], dtype='datetime64[us]', freq='ME')


## 15 选学：日历偏移与工作日
DateOffset(days=1) 按日历走到下一天的同一钟面时间；Timedelta(days=1) 则总是经过 24 小时。跨夏令时切换时，两者可能不同。以下日期取自官方文档的 Helsinki 回拨示例。

In [17]:
before = pd.Timestamp("2016-10-30 00:00", tz="Europe/Helsinki")
fixed = before + pd.Timedelta(days=1)
calendar = before + pd.DateOffset(days=1)
print(fixed)
print(calendar)
print((calendar - before).total_seconds() / 3600)
# 24 小时后仍是 10-30 23:00；下一个日历日的 00:00 则相隔 25 小时。

2016-10-30 23:00:00+02:00
2016-10-31 00:00:00+02:00
25.0


BusinessDay 默认按周一至周五移动。CustomBusinessDay 可通过 weekmask 指定每周工作日，用 holidays 排除额外日期。

下面只定义一个自制工作日历，把 2026-01-05 设为休息日；它不代表任何地区的法定节假日安排。真实业务须提供自己的日历规则。

In [18]:
friday = pd.Timestamp("2026-01-02")
custom_day = pd.offsets.CustomBusinessDay(
    weekmask="Mon Tue Wed Thu Fri", holidays=["2026-01-05"],
)
print(friday + pd.offsets.BusinessDay())
print(friday + custom_day)
# 普通工作日偏移得到周一 01-05；自制日历再跳过该日，得到周二 01-06。

2026-01-05 00:00:00
2026-01-06 00:00:00


## 本章小结

（1）先明确日期格式、数字单位与起点；保留原始输入，用 NaT 和失败标记追踪转换问题。

（2）用 Timestamp 表示单个时间，用 DatetimeIndex 组织时间标签；dt 提取字段，loc 按时间范围筛选。

（3）tz_localize 确定本地时间所属时区，tz_convert 改变同一时刻的显示时区；夏令时歧义与缺口需要明确规则。

（4）查看实际时间单位，检查精度和范围；总耗时用 total_seconds，日历移动与固定时长分别选择。

## 练习

（1）将下面的结束时间转为日期类型，保留转换失败记录，并计算有效任务的分钟数。结果中仍要保留原始行标签。

In [19]:
practice = pd.DataFrame({
    "start": ["2026-09-20 10:00", "2026-09-20 11:00"],
    "end": ["2026-09-20 10:45", "invalid"],
}, index=["P1", "P2"])
# 补充：显式给出格式，生成新日期列、失败标记和分钟数。
# 检查：P1 为 45 分钟；P2 的失败可定位，原文本与 P1、P2 标签仍在。

（2）先预测下面代码的时间点个数、起止时间和两种秒数，再运行。说明 inclusive 与 seconds 分别控制或表示什么。

In [20]:
prediction = pd.date_range(
    "2026-09-20 09:00", "2026-09-20 10:00",
    freq="20min", inclusive="right", unit="s",
)
span = pd.Timedelta(days=1, minutes=1)
print(prediction)
print(span.seconds, span.total_seconds())
# 补充：记录预测，并解释是否包含 09:00 和 10:00。

DatetimeIndex(['2026-09-20 09:20:00', '2026-09-20 09:40:00',
               '2026-09-20 10:00:00'],
              dtype='datetime64[s]', freq='20min')
60 86460.0


（3）输入原先都是上海本地时间，现在新增一组自带 UTC 偏移的记录。两组输入分别应选择 tz_localize 还是 to_datetime(..., utc=True)？说明理由，把它们统一到 UTC，并比较是否表示同一时刻。

In [21]:
local_input = ["2026-09-20 09:00"]
offset_input = ["2026-09-20 01:00 +0000"]
# 补充：分开解析两种输入，避免把上海的 09:00 直接解释为 UTC。
# 检查：统一后两条记录相等；在注释中说明输入条件改变后的方法选择。

（4）两条 CET 本地记录都写为 2018-10-28 02:30，但人工日志注明前者是夏令时，后者是标准时。请据此本地化，计算真实间隔。如果人工标记丢失，说明为何不能直接判为零时长，以及你会怎样保留不确定记录。

In [22]:
same_clock = pd.to_datetime(["2018-10-28 02:30", "2018-10-28 02:30"])
dst_flags = [True, False]
# 补充：使用人工标记处理 ambiguous，并比较 UTC 时刻和间隔。
# 检查：实际间隔为 1 小时；缺少标记时说明报错或标为 NaT 的选择理由。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方文档 | 核查版本 pandas 3.0.6。[to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) 的 Parameters、Returns、Timezones：format、errors、unit、origin、utc 与混合偏移；[Timestamp](https://pandas.pydata.org/docs/reference/api/pandas.Timestamp.html)、[DatetimeIndex](https://pandas.pydata.org/docs/reference/api/pandas.DatetimeIndex.html)、[date_range](https://pandas.pydata.org/docs/reference/api/pandas.date_range.html) 的类型、freq、inclusive、unit；[Series.dt](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.html) 与 [Time series](https://pandas.pydata.org/docs/user_guide/timeseries.html#partial-string-indexing) 的 Partial string indexing、Timestamp limitations、DateOffset objects、Custom business days、Time span representation；[3.0 时间精度推断变化](https://pandas.pydata.org/docs/whatsnew/v3.0.0.html#datetime-timedelta-resolution-inference) 与 [Timestamp.as_unit](https://pandas.pydata.org/docs/reference/api/pandas.Timestamp.as_unit.html) 的单位、范围及 round_ok；[tz_localize](https://pandas.pydata.org/docs/reference/api/pandas.DatetimeIndex.tz_localize.html) 的 ambiguous、nonexistent、Raises 与 CET/Warsaw 示例，[tz_convert](https://pandas.pydata.org/docs/reference/api/pandas.DatetimeIndex.tz_convert.html) 的转换与无时区错误；[to_timedelta](https://pandas.pydata.org/docs/reference/api/pandas.to_timedelta.html) 的 unit、errors 和月份限制，[Time deltas](https://pandas.pydata.org/docs/user_guide/timedeltas.html#attributes) 的 Attributes、components，[total_seconds](https://pandas.pydata.org/docs/reference/api/pandas.Timedelta.total_seconds.html)；[Period](https://pandas.pydata.org/docs/reference/api/pandas.Period.html) 的区间、频率与 start_time。 |
| Python 官方文档 | Python 3.12.14 [datetime：strftime() and strptime() Format Codes](https://docs.python.org/3.12/library/datetime.html#strftime-and-strptime-format-codes)：%Y、%m、%d、%H、%M、%z 的含义；pandas 支持的小数秒精度另按 pandas 文档。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[timeseries](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/timeseries.rst)、[v3.0.0](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/whatsnew/v3.0.0.rst)、[timedeltas](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/timedeltas.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |